In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
sheet_config = {
    "A - Daily": {"id_cols": ["Date"]},
    "B - Daily": {"id_cols": ["Date"]},
    "C - Daily": {"id_cols": ["Date"]},
    "D - Daily": {"id_cols": ["Date"]},
    "A - Interval": {"id_cols": ["Month", "Day", "Interval"]},
    "B - Interval": {"id_cols": ["Month", "Day", "Interval"]},
    "C - Interval": {"id_cols": ["Month", "Day", "Interval"]},
    "D - Interval": {"id_cols": ["Month", "Day", "Interval"]},
    "Daily Staffing": {"id_cols": ["Unnamed: 0"]},
}


In [3]:
def choose_arima_order(series, max_p=4, max_q=2, max_d=2):
    """
    Pick a simple ARIMA order using ADF test for d
    and AIC for p and q.
    """
    s = series.dropna()

    if len(s) < 20:
        return (1, 0, 0)

    d = 0
    temp = s.copy()

    for _ in range(max_d):
        try:
            p_value = adfuller(temp)[1]
            if p_value < 0.05:
                break
            temp = temp.diff().dropna()
            d += 1
        except:
            break

    best_order = (1, d, 1)
    best_aic = np.inf

    for p in range(max_p + 1):
        for q in range(max_q + 1):
            if p == 0 and q == 0:
                continue

            try:
                model = SARIMAX(
                    s,
                    order=(p, d, q),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                result = model.fit(disp=False)

                if result.aic < best_aic:
                    best_aic = result.aic
                    best_order = (p, d, q)
            except:
                continue

    return best_order

In [4]:
def kalman_fill(series, order=None):
    """
    Fill missing values using SARIMAX + Kalman smoothing.
    If that fails, use interpolation instead.
    """
    if series.isna().sum() == 0:
        return series.copy()

    if order is None:
        order = choose_arima_order(series)

    try:
        model = SARIMAX(
            series,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        result = model.fit(disp=False)

        smoothed_values = pd.Series(
            result.smoother_results.smoothed_forecasts[0],
            index=series.index
        )

        filled = series.copy()
        filled[series.isna()] = smoothed_values[series.isna()]
        return filled

    except Exception as e:
        print("Kalman smoothing failed, using interpolation instead:", e)
        return series.interpolate().bfill().ffill()


In [5]:
def impute_one_sheet(sheet_name, df, verbose=True):
    """
    Impute missing numeric columns in one sheet.
    """
    if sheet_name not in sheet_config:
        if verbose:
            print(f"{sheet_name} not found in config, returning original data.")
        return df.copy()

    id_cols = sheet_config[sheet_name]["id_cols"]
    new_df = df.copy()

    for col in df.columns:
        if col in id_cols:
            continue

        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        if df[col].isna().sum() == 0:
            continue

        missing_count = df[col].isna().sum()

        if verbose:
            print(f"[{sheet_name}] Working on column: {col}")
            print(f"Missing values: {missing_count}")

        order = choose_arima_order(df[col])

        if verbose:
            print(f"Chosen ARIMA order: {order}")

        filled_col = kalman_fill(df[col], order=order)

        observed_min = df[col].min()
        observed_max = df[col].max()
        filled_col = filled_col.clip(lower=observed_min, upper=observed_max)

        if filled_col.isna().sum() > 0:
            if verbose:
                print(f"Still missing values in {col}, using forward/backward fill.")
            filled_col = filled_col.ffill().bfill()

        new_df[col] = filled_col
    return new_df


In [6]:
def impute_all_sheets(datasets):
    final_data = {}
    print("I am Here!")
    for sheet_name, df in datasets.items():
        if sheet_name in sheet_config:
            final_data[sheet_name] = impute_one_sheet(sheet_name, df)
        else:
            final_data[sheet_name] = df.copy()

    # quick check for remaining NaNs
    print("\nChecking remaining missing values...")
    for sheet_name, df in final_data.items():
        total_missing = df.select_dtypes(include=[np.number]).isna().sum().sum()
        print(f"{sheet_name}: {total_missing} missing values")


    print("Saving CSV files to")

    for sheet_name, df in final_data.items():
        safe_name = sheet_name.replace(" ", "_").replace("-", "")+ ".csv"
        df.to_csv(safe_name, index=False)

    print("Done saving files.")

    return final_data


if __name__ == "__main__":
    print("Load your Excel file into a dictionary first, then run impute_all_sheets(datasets).")

Load your Excel file into a dictionary first, then run impute_all_sheets(datasets).
